### 라이브러리

In [1]:
import pandas as pd
import numpy as np

### 데이터 호출

In [2]:
df = pd.read_csv('train.csv')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 908 entries, 0 to 907
Columns: 2881 entries, PRODUCT_ID to X_2875
dtypes: float64(2876), int64(1), str(4)
memory usage: 20.0 MB


### STEP1 제품별 분리 

In [3]:
def split_by_product(df, product_col='PRODUCT_CODE'):
    df_A = df[df[product_col] == 'A_31'].copy()
    df_TO = df[df[product_col].isin(["T_31","O_31"])].copy()

    return df_A, df_TO 

### STEP2 누수 컬럼 제거 

In [4]:
def drop_leakage(df, 
                 target_col='Y_Quality',
                 class_col='Y_Class',
                 id_cols = ['PRODUCT_ID', 'PRODUCT_CODE'],
                 time_cols = 'TIMESTAMP'):
    y = df[target_col].copy()
    y_class = df[class_col].copy()
    temp = df[id_cols + [time_cols]].copy()
    X = df.drop(columns=[target_col, class_col] + id_cols + [time_cols])
    return X, y, y_class, temp

### STEP3 위장 결측 -> NaN

In [5]:
def missing_value(X, placeholder = [999,9999,-999,-9999]):
    X = X.replace(placeholder, np.nan)
    return X

### STEP4 완전 결측 컬럼 제거

In [6]:
def remove_all_nan(X):
    all_nan_cols = X.columns[X.isna().all()].tolist()
    X = X.drop(columns=all_nan_cols)
    return X, all_nan_cols

### STEP5 결측 비율 60% 초과 컬럼 제거

In [7]:
def remove_high_missing(X, threshold=0.6):
    missing_ratio = X.isna().mean()
    drop_cols = missing_ratio[missing_ratio > threshold].index.tolist()
    X = X.drop(columns=drop_cols)
    return X, drop_cols

### STEP6 상수/제로분산 제거

In [8]:
def remove_constant(X):
    nunique = X.columns[X.nunique(dropna=False) <= 1]
    std_zero = X.select_dtypes(include="number").columns[
        X.select_dtypes(include="number").std() == 0
    ]
    
    drop_cols = list(set(nunique) | set(std_zero))
    X = X.drop(columns=drop_cols)
    
    return X, drop_cols

### STEP7 저분산 제거

In [9]:
def remove_low_variance(X, threshold=0.01):
    num_cols = X.select_dtypes(include="number").columns
    if len(num_cols) == 0:
        return X, []

    X_num = X[num_cols]
    col_range = (X_num.max() - X_num.min()).replace(0, np.nan)  # 상수 컬럼은 STEP4에서 이미 제거됨

    X_scaled = (X_num - X_num.min()) / col_range
    low_var_cols = X_scaled.columns[X_scaled.var(skipna=True) < threshold].tolist()

    X = X.drop(columns=low_var_cols)
    return X, low_var_cols

### STEP8 상관계수 높은 컬럼 제거

In [10]:
def remove_high_correlation(X, threshold=0.95):
    num_cols = X.select_dtypes(include="number").columns
    corr_matrix = X[num_cols].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    drop_cols = [col for col in upper.columns if (upper[col] > threshold).any()]

    X = X.drop(columns=drop_cols)
    return X, drop_cols

### STEP9 라인 인코딩

In [11]:
def encode_line(X, line_col='LINE'):
    if line_col in X.columns:
        X = pd.get_dummies(X, columns=[line_col], drop_first=True)
    return X

### 전체 전처리 함수 

In [12]:
def preprocess(
    df,
    target_col='Y_Quality',
    class_col='Y_Class',
    id_cols=['PRODUCT_ID', 'PRODUCT_CODE'],
    time_cols='TIMESTAMP',
    placeholder=[999, 9999, -999, -9999],
    line_col='LINE',
    missing_threshold=0.6,
    variance_threshold=0.01,
    corr_threshold=0.95,
    remove_nan=True
):
    # STEP2 누수 컬럼 제거
    X, y, y_class, temp = drop_leakage(
        df,
        target_col=target_col,
        class_col=class_col,
        id_cols=id_cols,
        time_cols=time_cols
    )

    # STEP3 위장결측 → NaN (결측/분산 판단 정확도를 위해 다른 결측·분산 처리보다 먼저 수행)
    X = missing_value(X, placeholder)

    # STEP4 완전 결측 컬럼 제거
    if remove_nan:
        X, all_nan_cols = remove_all_nan(X)
    else:
        all_nan_cols = X.columns[X.isna().all()].tolist()

    # STEP5 결측 비율 60% 초과 컬럼 제거
    X, high_missing_cols = remove_high_missing(X, missing_threshold)

    # STEP6 상수/제로분산 제거
    X, constant_cols = remove_constant(X)

    # STEP7 저분산 제거
    X, low_var_cols = remove_low_variance(X, variance_threshold)

    # STEP8 상관계수 높은 컬럼 제거
    X, high_corr_cols = remove_high_correlation(X, corr_threshold)

    # STEP9 LINE 인코딩
    X = encode_line(X, line_col)

    print("=" * 50)
    print(f"최종 데이터 크기 : {X.shape}")
    print(f"완전결측 제거 : {len(all_nan_cols)}개")
    print(f"결측비율 {missing_threshold*100:.0f}% 초과 제거 : {len(high_missing_cols)}개")
    print(f"상수/제로분산 제거 : {len(constant_cols)}개")
    print(f"저분산 제거 : {len(low_var_cols)}개")
    print(f"고상관 제거 : {len(high_corr_cols)}개")
    print("=" * 50)

    return X, y, y_class, temp

### 전처리 적용

In [13]:
df_A, df_TO = split_by_product(df)
X_A, y_A, yclass_A, temp_A = preprocess(df_A)
X_TO, y_TO, yclass_TO, temp_TO = preprocess(df_TO)

print(X_A.shape)

print(X_TO.shape)

최종 데이터 크기 : (316, 551)
완전결측 제거 : 683개
결측비율 60% 초과 제거 : 612개
상수/제로분산 제거 : 225개
저분산 제거 : 43개
고상관 제거 : 764개


최종 데이터 크기 : (592, 334)
완전결측 제거 : 2195개
결측비율 60% 초과 제거 : 14개
상수/제로분산 제거 : 112개
저분산 제거 : 45개
고상관 제거 : 176개
(316, 551)
(592, 334)


### CSV 형식으로 저장

In [14]:
df_A_processed = pd.concat([X_A, y_A, yclass_A, temp_A], axis=1)
df_A_processed.to_csv("A_31_preprocessed.csv", index=False, encoding="utf-8-sig")

df_TO_processed = pd.concat([X_TO, y_TO, yclass_TO, temp_TO], axis=1)
df_TO_processed.to_csv("TO_31_preprocessed.csv", index=False, encoding="utf-8-sig")